# Netflix / Streaming İçerik Analizi

Bu projede platformlardaki filmleri inceleyeceğim. Ayrıca IMDb puanını 3 modelle tahmin etmeyi de denedim.


In [ ]:
import pandas as pd
pd.set_option('display.max_columns',100)
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/MoviesOnStreamingPlatforms_updated.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df[['Netflix','Hulu','Prime Video','Disney+']].sum().plot(kind='bar')
plt.title('platform')
plt.show()


In [ ]:
df['Year'].value_counts().sort_index().plot()
plt.show()


In [ ]:
df['Genres'].fillna('').str.split(',').explode().str.strip().value_counts().head(10).plot(kind='barh')
plt.show()


### Boş veri


In [ ]:
df['IMDb']=df['IMDb'].fillna(df['IMDb'].median())
df['Runtime']=df['Runtime'].fillna(df['Runtime'].median())
df['Age']=df['Age'].fillna('Unknown')


### Feature Engineering


In [ ]:
x=df[['Year','Runtime','Netflix','Hulu','Prime Video','Disney+']].copy()
x['Age']=df['Age']
x=pd.get_dummies(x,drop_first=True)
y=df['IMDb']


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

for ad,m in [('LR',LinearRegression()),('DT',DecisionTreeRegressor(max_depth=8,random_state=42)),('RF',RandomForestRegressor(n_estimators=80,random_state=42))]:
    m.fit(x_train,y_train)
    print(ad,round(r2_score(y_test,m.predict(x_test)),3))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(n_estimators=80,random_state=42).fit(x_train,y_train)
pd.Series(rf.feature_importances_,index=x.columns).sort_values().tail(8).plot(kind='barh')
plt.show()
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


### Sonuç

Prime'da daha çok içerik var. IMDb tahmini zayıf çünkü puanı yıl/süre tek başına açıklamıyor. EDA hedefini tutturdum, puan modeli orta.
